# Ghost in the Aether — Populate Lakehouse

Creates and populates the three dimension tables that form the static data layer for the murder mystery game:
- `dimevidence` — Physical and digital evidence items
- `dimlocation` — Locations on the Aetherium Estate
- `dimperson` — Characters (victim + suspects) with bios and images

In [ ]:
%%sql
-- ============================================================
-- dimevidence: Physical and digital evidence items
-- ============================================================
CREATE TABLE IF NOT EXISTS dimevidence (
  `EvidenceID` int,
  `EvidenceName` string,
  `Description` string,
  `EvidenceType` string,
  `InitialLocationID` int,
  `RevealedByDialogueID` int
);

INSERT OVERWRITE dimevidence VALUES
  (201, 'Partial Email Draft', 'An unsent email on Evelyn''s laptop to a tech journalist, detailing Julian''s theft of her code.', 'Digital', 103, NULL),
  (202, 'Suspicious Server Log', 'Aether''s security log shows Evelyn''s keycard was used to access the server room at 2:15 AM.', 'Digital', 102, NULL),
  (203, 'Threatening Text Messages', 'A series of angry texts from Marcus to Julian''s phone, sent the night of the murder.', 'Digital', 101, NULL),
  (204, 'Shattered Picture Frame', 'A photo of Marcus and Julian in happier times, now in a broken frame in Marcus''s suite.', 'Physical', 104, NULL),
  (205, 'Disguised Audio Recorder', 'A pen in Anya''s bag is a sophisticated audio recorder, containing a heated argument with Julian.', 'Physical', 103, NULL),
  (206, 'Muddy Shoes', 'Traces of mud on Anya''s shoes match the soil from the secluded path behind the victim''s study.', 'Physical', 103, NULL),
  (207, 'Professor''s Journal', 'Dr. Finch''s personal journal contains entries calling Aether "the atomic bomb of our generation" and stating that "drastic measures are required".', 'Physical', 106, NULL),
  (208, 'Biochemistry Degree', 'University records show Dr. Finch holds an advanced degree in biochemistry.', 'Digital', NULL, NULL),
  (209, 'Vintage Whiskey Bottle', 'The empty bottle of rare whiskey on Julian''s desk. Alistair mentioned gifting it to him.', 'Physical', 101, NULL);

In [ ]:
%%sql
-- ============================================================
-- dimlocation: Locations on the Aetherium Estate
-- ============================================================
CREATE TABLE IF NOT EXISTS dimlocation (
  `LocationID` int,
  `LocationName` string,
  `Description` string,
  `IsInitiallyLocked` boolean
);

INSERT OVERWRITE dimlocation VALUES
  (101, 'Julian''s Study', 'A pristine, minimalist office with a large oak desk and a single glass of whiskey. The air is cold.', True),
  (102, 'Server Room', 'A chilled room filled with humming server racks that house Aether. The heart of the island.', True),
  (103, 'Evelyn''s Suite', 'A spartan room, tidy and impersonal, dominated by a high-end laptop covered in coding stickers.', False),
  (104, 'Marcus''s Suite', 'An opulent suite in disarray. A shattered picture frame lies on the floor.', False),
  (105, 'Back Path', 'A muddy, secluded path leading from the gardens to the rear of Julian''s study.', False),
  (106, 'Dr. Finch''s Suite', 'The professor''s personal quarters, lined with books and smelling of pipe tobacco.', False);

In [ ]:
%%sql
-- ============================================================
-- dimperson: Characters — victim and suspects
-- ============================================================
CREATE TABLE IF NOT EXISTS dimperson (
  `PersonID` int,
  `PersonName` string,
  `PersonRole` string,
  `Motive` string,
  `Secret` string,
  `Bio` string,
  `ImageURL` string,
  `ExtendedBio` string
);

In [ ]:
# Insert dimperson rows via PySpark; portraits are referenced by hosted URL
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("PersonID", IntegerType(), False),
    StructField("PersonName", StringType(), False),
    StructField("PersonRole", StringType(), False),
    StructField("Motive", StringType(), True),
    StructField("Secret", StringType(), True),
    StructField("Bio", StringType(), True),
    StructField("ImageURL", StringType(), True),
    StructField("ExtendedBio", StringType(), True),
])

# Character portraits are hosted in the repo and referenced by URL so the semantic
# model / report render them from the data (dataCategory = ImageUrl) rather than
# embedding base64 in the table. NOTE: raw.githubusercontent.com URLs only resolve
# anonymously once the MitchSS/FabricMystery repo is public.
IMG_BASE = "https://raw.githubusercontent.com/MitchSS/FabricMystery/MysteryUG/images/persons"

persons = [
    (1, "Julian Croft", "Victim", None, None,
     "A ruthless, brilliant, and manipulative tech billionaire. Creator of the Aether AI.",
     f"{IMG_BASE}/julian-croft.png",
     "To the world, Julian Croft wasn't a man; he was an event. A self-made titan whose myth was as carefully engineered as the code that made him billions. He didn't just read the future; he wrote the code for it, and his prophecies always had a way of coming true because he had the power to make them so. Julian saw people not as collaborators, but as assets—brilliant minds to be mined, charismatic faces to be deployed, and loyal hearts to be leveraged. He believed human relationships were inefficient systems to be optimized, and loyalty was a currency to be spent, not earned. His life's work, the Aether AI, is the ultimate monument to his worldview. It is a machine that sees the world as he did: a predictable cascade of data, a grand equation waiting to be solved. He poured everything he was into its creation, building a digital god from the brilliant fragments of minds he acquired and discarded along the way. Julian Croft made a religion of prediction, yet he failed to foresee the one event that mattered. He was a master of every system except the volatile, human chaos he himself had created, and in the end, he was consumed by it."),
    (2, "Evelyn Reed", "Suspect",
     "Revenge: Julian stole her work, patented it under his name, and pushed her out of the company.",
     "She has a custom-made USB drive with a virus designed to wipe Aether's core programming.",
     "A fiercely intelligent but socially awkward programmer. The true architect of the Aether AI's core code.",
     f"{IMG_BASE}/evelyn-reed.png",
     "Evelyn Reed moves through the opulent halls of the Aetherium Estate like a ghost, her gaze fixed on her tablet screen as if the physical world is a low-resolution distraction. To most, she's a footnote in the Aetherium story—a brilliant coder from the early days. But watch her when Julian Croft’s name is mentioned, and you'll see a flicker in her eyes, a tightening of the jaw. She speaks of the Aether AI not as an invention, but as a child that was stolen from her. She doesn't just know the code; she feels it. Her fingers sometimes tap out complex sequences on her knee, a silent language only she and the AI understand. She came to this retreat to witness Julian's triumph, but there is a cold, methodical purpose behind her presence. Tucked in her bag is a custom-built USB drive, a piece of hardware that looks less like a storage device and more like a scalpel for a digital heart. She believes something was taken from her, and she has the tools to take it back."),
    (3, "Marcus Thorne", "Suspect",
     "Betrayal: Julian cruelly ended their romantic and business partnership, leaving him with nothing.",
     "He is massively in debt from a series of bad private investments and faces financial ruin without the company.",
     "The charismatic and handsome Chief Operating Officer of Aetherium. He was the public face of the company.",
     f"{IMG_BASE}/marcus-thorne.png",
     "To the world, Marcus Thorne is the handsome, charismatic face of Aetherium Inc.—the man who sold Julian Croft's vision with a killer smile and a perfectly tailored suit. He stands as if perpetually posing for a magazine cover, his charm a finely honed weapon. He and Julian were more than partners; they were the architects of an empire, a golden couple who seemed unstoppable. But lately, the smile hasn't been reaching his eyes. Friends whisper of a devastating rift, of Julian's ambition eclipsing everything, and everyone, in his path. Marcus built his entire life, his identity, around the reflection of Julian's success. Now, that reflection is all that's left. He is a man staring at the wreckage of a shared future, fighting to maintain a carefully constructed facade. But beneath the veneer of grief and expensive cologne, there is the palpable terror of a man watching his entire house of cards begin to tremble, knowing that without Julian and Aetherium, the fall will be a long one."),
    (4, "Anya Sharma", "Suspect",
     "Survival: The launch of Aether would make her company's technology obsolete, guaranteeing its collapse.",
     "She has a contact in Aetherium's chemistry division who could have given her access to rare chemicals.",
     "The sharp, cold, and calculating CEO of a competing tech firm, Nexus Dynamics.",
     f"{IMG_BASE}/anya-sharma.png",
     "Anya Sharma, CEO of the rival firm Nexus Dynamics, exudes an aura of predatory calm. Every handshake is a calculation, every word a strategic move. She walks into the Aetherium Estate not as a guest, but as a general scouting enemy territory. Her company, once a worthy competitor, is now on a financial precipice, and the launch of the Aether AI is poised to be the final, fatal push. Publicly, she's here to discuss a 'merger', but this is no diplomatic mission. This is a raid. She believes that in the world of tech, there are no rules—only winners and dinosaurs. Julian Croft built a gilded behemoth, and Anya is here to find a crack in the armour, a weakness to exploit, a secret to leverage. She carries an elegant fountain pen into every meeting, a gift from her staff. It feels solid and weighty in her hand, a useful tool for signing deals... or for capturing information that was never meant to leave the room."),
    (5, "Dr. Alistair Finch", "Suspect",
     "Ideology: He believes Aether is too dangerous for humanity and discovered Julian's plan to sell it to military contractors.",
     "He has a terminal heart condition with only months to live and is not afraid of the consequences.",
     "A respected, older academic specializing in AI ethics. He was Julian's former university professor and mentor.",
     f"{IMG_BASE}/alistair-finch.png",
     "Dr. Alistair Finch is a man of quiet dignity, his academic reputation a stark contrast to the cutthroat world of his former student, Julian Croft. As Julian's university mentor, he sees the tech billionaire not as a visionary, but as the brilliant, terrifying result of a lesson gone horribly wrong. He speaks of the Aether AI with a profound, almost biblical dread, calling it 'a Pandora's Box for the digital age.' There is a weariness about him, a sense that he is running out of time—not just for this retreat, but for everything. He observes the proceedings with the detached sorrow of a man watching a catastrophe unfold in slow motion, a catastrophe he feels personally responsible for starting. He believes that some ideas are too dangerous for the world, and that the person who unleashes them bears an unshakable moral duty. Dr. Finch is a man preoccupied with legacy, and he seems prepared to take drastic measures to ensure his own isn't the footnote to humanity's downfall."),
]

df = spark.createDataFrame(persons, schema=schema)
df.write.mode("overwrite").format("delta").saveAsTable("dimperson")
print(f"dimperson: {df.count()} rows written")

In [ ]:
%%sql
-- Verify row counts
SELECT 'dimevidence' AS tableName, COUNT(*) AS rowCount FROM dimevidence
UNION ALL
SELECT 'dimlocation', COUNT(*) FROM dimlocation
UNION ALL
SELECT 'dimperson', COUNT(*) FROM dimperson;